# Day 9: Baseline Logistic Regression Model
Train-Test Split & Evaluation using a cleaned dataset from Week 1.

## Objective
- Load the cleaned dataset
- Create a binary target variable
- Split data 80/20 without leakage
- Train a Logistic Regression model
- Evaluate accuracy on unseen test data
  

## Step 1  Import Libraries
- We import only what we need. No scaling libraries yet, scaling before split causes data leakage.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Step 2  Load the Cleaned Dataset
This is the `model_ready_features.csv` file produced during Week 1 cleaning.

In [2]:
df = pd.read_csv('model_ready_features.csv')
df.head()

,User_ID,Age,Score,Month,DayOfWeek,Is_Weekend,Dept_HR,Dept_Marketing,Dept_Sales,City_New York,City_Paris,City_Tokyo
0,1,34.967142,65.458919,1,6,1,0,0,1,0,0,0
1,2,28.617357,32.669678,1,0,0,1,0,0,1,0,0
2,3,36.476885,58.631558,1,1,0,0,0,0,0,1,0
3,4,45.230299,40.711423,1,2,0,0,0,1,0,0,1
4,5,27.658466,45.088958,1,3,0,0,0,1,0,0,1


## Step 3  Create Binary Target Variable
Logistic Regression requires a binary target. We create `Is_High_Score`: 1 if Score > 50, else 0.

In [3]:
df['Is_High_Score'] = (df['Score'] > 50).astype(int)
df['Is_High_Score'].value_counts()

Is_High_Score
1    51270
0    46294
Name: count, dtype: int64

## Step 4  Separate Features (X) and Target (y)
We drop the original `Score` column and the new target from X to avoid leakage.

In [4]:
X = df.drop(columns=['Score', 'Is_High_Score'])
y = df['Is_High_Score']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (97564, 11)
Target shape: (97564,)


## Step 5  Train-Test Split (80/20)
We split BEFORE any fitting or scaling. Using `random_state=42` for reproducibility.

Why split first? Fitting a scaler on the full dataset leaks test statistics into training.
The model indirectly sees test data, which inflates accuracy and gives a false reading.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

Training samples: 78051
Test samples:     19513


## Step 6 Train the Logistic Regression Model
We train only on X_train and y_train. The test set is untouched at this stage.

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model trained successfully.")

Model trained successfully.


## Step 7  Predict on Test Data
We generate predictions on X_test data the model has never seen.

In [7]:
y_pred = model.predict(X_test)
print("Predictions:", y_pred[:10])
print("Actual:     ", y_test.values[:10])

Predictions: [1 1 1 1 1 1 1 1 1 1]
Actual:      [1 1 1 1 0 1 0 1 0 1]


## Step 8  Evaluate Accuracy
Accuracy = correct predictions / total test samples. This is our baseline metric.

In [8]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.2f}")

Model Accuracy on Test Set: 0.53


In [9]:
print(df['Is_High_Score'].value_counts(normalize=True))
print(df.shape)
print(X.columns.tolist())

Is_High_Score
1    0.525501
0    0.474499
Name: proportion, dtype: float64
(97564, 13)
['User_ID', 'Age', 'Month', 'DayOfWeek', 'Is_Weekend', 'Dept_HR', 'Dept_Marketing', 'Dept_Sales', 'City_New York', 'City_Paris', 'City_Tokyo']


## Conclusion

### Results
- Model Accuracy: 0.53 (53%)

### Why is accuracy low?
The dataset used is synthetically generated. The features (Age, Department, 
City, DayOfWeek etc.) were created independently with no statistical 
relationship to the Score column. Because of this, the model has no real 
signal to learn from, and 53% accuracy is the expected outcome, barely 
above random guessing on a balanced dataset.

### What this baseline tells us
- The data pipeline works correctly, split, train, predict, evaluate all 
  executed without data leakage
- The problem is data quality, not model failure
- A real world dataset with genuine feature-target relationships would 
  produce significantly higher accuracy

### Key Takeaways
- Always split before fitting, data leakage ban followed strictly
- Baseline models expose data quality issues early, saving wasted effort later
- Model performance is only as good as the data it learns from